# 02 — Silver: Inventory clean Silver (M4)

**Owner:** M4 • **Tier:** Light • **Phase:** 4 (Thu 14 May lunch – Sun 17 May)
**Hard deadline:** Day 5 EOD (Sat 16 May) — your clean Silver is needed by lead before T6 reconstruction can begin Day 6.

## Scope — T6 ATP reconstruction is LEAD's, not yours
- **Source tables (8):** `si_inventory_movements` (438K), `si_inventory_snapshots` (217K), `si_movement_types`, `wh_inbound_receipts`, `wh_inventory_movements`, `wh_inventory_snapshots`, `wh_movement_types`, `wh_warehouses`
- **Silver transforms owned:** apply T1/T2/T3 utilities + flag obvious anomalies (no T# of your own — T6 reconstruction belongs to lead)
- **Anomalies in your territory:** A6, A10 (2)

## Workflow split for inventory
- **You write:** `silver_store_inventory_snapshots` (clean source rows, no reconstruction), all wh_* Silver tables
- **Lead writes:** `silver_store_inventory_snapshots_reconstructed` (sibling table with synthesized gap-fill rows from T6)
- **Lead's `fact_store_inventory_snapshot`** UNIONs your clean Silver + lead's reconstructed Silver

## Hint script
- **A6 (negative qty):** physical_qty/sellable_qty should never go negative. Filter `si_inventory_snapshots` for negative values. Flag `NEGATIVE_QTY`.
- **A10 (open-box restocked as new):** in `rr_return_receipts` (you don't own — read from M5's Silver), find rows where `condition_on_receipt='OPENED'` AND `restocked_as_condition='NEW'`. Cross-reference how those flow into your inventory movements. Flag `OPEN_BOX_AS_NEW`.

## Hands-on-of-everything checklist
- [ ] Bronze read
- [ ] Silver write: 8 tables (clean) — T6 sections in this notebook are now LEAD's; ignore/skip them
- [ ] Gold dim: dim_store
- [ ] Gold fact: fact_warehouse_inventory_snapshot (gapless, easy)
- [ ] Anomalies: A6, A10
- [ ] Bus Matrix: 1 fact row + dim_store cells
- [ ] Report: Sections 1, 2, 7 (your T1/T2/T3 application), 8 (A6, A10), 9


## Setup — Install connector + widgets (Free Edition serverless)

_Brief Section 7.4 specified a Spark-Snowflake Maven JAR; Free Edition replaces this with the pure-Python `snowflake-connector-python`. See `docs/databricks_setup.md`._

In [ ]:
%pip install -q snowflake-connector-python pandas rapidfuzz
# dbutils.library.restartPython()  # SKIP on Free Edition — kernel hangs

In [ ]:
dbutils.widgets.text('sf_account',   'rhxendw-yb24678')
dbutils.widgets.text('sf_user',      'NEXAMART_LEAD')   # change to NEXAMART_M{2..6} for members
dbutils.widgets.text('sf_password',  '')                # paste at notebook run time
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role',      'ACCOUNTADMIN')    # NEXAMART_ENGINEER for members

## Cell 2 — Imports & lookup tables (movement types, warehouses)

In [ ]:
from pyspark.sql import functions as F, Window
import sys
sys.path.append('/Workspace/Repos/<your-org>/nexamart-m1/notebooks/_shared')
from utils_dates     import parse_date, is_parse_failure
from utils_keys      import surrogate_key, anonymous_key
from utils_anomaly   import add_anomaly_columns, flag, flag_date_parse_failure
from utils_snowflake import read_from_snowflake, write_to_snowflake

def read_bronze(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_BRONZE')

def read_silver(t):
    return read_from_snowflake(spark, t, schema='NEXAMART_SILVER')

def write_silver(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_SILVER')
    print(f'Wrote silver.{t} ({n} rows)')

def write_gold(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_GOLD')
    print(f'Wrote gold.{t} ({n} rows)')

## Cell 3 — `silver_inventory_movements` (store + warehouse unioned)

**TODO:**
- Read both `si_inventory_movements` (DD/MM/YYYY) and `wh_inventory_movements` (ISO with T)
- Parse dates with appropriate format hints (see `docs/date_formats.md`)
- Add a `source_system` column ('STORE' / 'WAREHOUSE')
- Generate SK on `(source_system, location_id, sku, movement_datetime, sequence)`
- Join `si_movement_types` and `wh_movement_types` for human-readable type names
- Add anomaly columns; flag DATE_PARSE_FAIL
- **Flag B3:** WH movements with NULL `reference_number` → `MOVEMENT_NULL_REF`, status `FLAGGED_AMBIGUOUS`
- Write `silver_inventory_movements`

In [ ]:
# TODO M4: implement Cell 3
# si_mov = read_bronze('si_inventory_movements')
# wh_mov = read_bronze('wh_inventory_movements')
# # Parse dates...
# # Union with source_system column...
# # flag(NULL reference_number, 'MOVEMENT_NULL_REF', 'FLAGGED_AMBIGUOUS', 'INFERRED')

## Cell 4 — `silver_warehouse_inventory_snapshots` (gapless — straightforward)

**TODO:**
- Read `wh_inventory_snapshots`
- Parse `snapshot_date` (ISO date format)
- SK on `(warehouse_id, sku, snapshot_date)`
- **Flag A5:** rows where `atp_qty > 0` AND `physical_qty = 0` → `ATP_POSITIVE_PHYSICAL_ZERO`, status `FLAGGED_ANOMALY`, certainty `UNRELIABLE`
- **Flag A6** (negative): any row with `physical_qty < 0` or `sellable_qty < 0` → `NEGATIVE_QTY`
- Write

In [ ]:
# TODO M4: implement Cell 4

## Cell 5 — `silver_store_inventory_snapshots` with T6 reconstruction

This is the hardest cell.

**Step 5a — Profile gaps:**
Build the cartesian product (store_id × calendar_day for the project window). Left-join actual snapshots. Where missing, those are gap days that need reconstruction.

**Step 5b — For each (store, sku, gap_day):**
1. Find the latest source snapshot before the gap (`last_known_snapshot`)
2. Sum movements between last snapshot and gap day, partitioned by movement type (`receives`, `issues`, `damages`, `return_receives`)
3. Compute `ATP_reconstructed`
4. Emit row with `data_quality_status='RECONSTRUCTED'`, `metric_certainty='INFERRED'`, code `RECONSTRUCTED_SNAPSHOT`

**Step 5c — Union real + reconstructed**, then apply A6 flag.

**Step 5d — Apply A10 detection (M4 scope, OPEN_BOX_AS_NEW):** Check `rr_return_receipts` for rows where `condition_on_receipt='OPENED'` AND `restocked_as_condition='NEW'`. Expect **12 receipts**. For each such receipt, the corresponding inventory movement that increases NEW stock is anomalous. Flag with `OPEN_BOX_AS_NEW`, status `FLAGGED_ANOMALY`, certainty `UNRELIABLE`.

> **NOT your anomaly:** A7 (`RESTOCK_BEFORE_INSPECTION`, predicate `inspection_status='PENDING' AND restocked_qty > 0`, expect **10 receipts**) is **Lead's** scope per `docs/tasks/lead.md` Task 23b — it's a different predicate on the same `rr_return_receipts` table. The two sets overlap but are not equal. Coordinate with Lead in the standup channel so the same receipt row isn't double-flagged on the Silver write.

In [ ]:
# TODO M4: implement Cell 5
# 5a — gap detection
# from pyspark.sql.functions import sequence, explode
# stores = read_bronze('pos_stores').select('store_id').distinct()
# date_spine = spark.range(1).select(
#     explode(sequence(F.lit('2024-03-01').cast('date'), F.lit('2024-08-31').cast('date'), F.expr('interval 1 day'))).alias('snapshot_date')
# )
# expected = stores.crossJoin(date_spine)
# real = read_bronze('si_inventory_snapshots') # parse date first
# # gaps = expected.left_anti_join(real on store_id+date)
# # ...
# # 5d — A10 detection ONLY (A7 lives in lead's notebook)
# returns = read_bronze('rr_return_receipts')
# bad_restock = returns.filter(
#     (F.col('condition_on_receipt') == 'OPENED') & (F.col('restocked_as_condition') == 'NEW')
# )
# # join to inventory movements that resulted from those receipts and flag OPEN_BOX_AS_NEW

## Cell 6 — `silver_inbound_receipts`

**TODO:**
- Read `wh_inbound_receipts`
- Parse date
- SK on `receipt_id`
- Add anomaly cols (mostly CLEAN; flag NEGATIVE_QTY if any qty col is negative)
- Write

In [ ]:
# TODO M4: implement Cell 6

## Cell 7 — Acceptance check

Before requesting PR review:
1. Reconstructed rows are flagged distinguishably (`data_quality_status='RECONSTRUCTED'`)
2. The 3 problem stores are reconstructed for ~7 days each in the campaign window (you'll see)
3. Negative-qty rows exist and are flagged `NEGATIVE_QTY` (small number)
4. ATP-positive-physical-zero rows are flagged `ATP_POSITIVE_PHYSICAL_ZERO` (small number, in WH only)
5. Open-box-restocked-as-new rows flagged `OPEN_BOX_AS_NEW`
6. WH movements with NULL ref flagged `MOVEMENT_NULL_REF` (single-digit hundreds)
7. All 4 mandatory anomaly columns populated on every row, every Silver table

In [ ]:
# Verification queries
# inv = spark.read.format('snowflake').options(**sf_silver).option('dbtable', 'silver_store_inventory_snapshots').load()
# inv.groupBy('data_quality_status').count().show()
# inv.groupBy('anomaly_reason_code').count().orderBy(F.desc('count')).show(truncate=False)